# WR-HBP Final Outer-Test Confirmation V1 — Kaggle

One-shot inference-only confirmation of frozen R0-HBP vs WR-HBP checkpoints. **No retraining or post-test tuning.**

In [ ]:
NOTEBOOK_BUILD = "WR-HBP-FINAL-OUTER-CONFIRMATION-V1"
SCIENTIFIC_CODE_COMMIT = "b62963faec8e337359ba4244a3ee5814709bc18f"
DEVELOPMENT_CODE_COMMIT = "01c9212965bc9040ef151204b9404d564f523a0f"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wr-final"
PROJECT = WORK / "wr-hbp-final-confirmation-v1-project"
OUT = PROJECT / "outer"
AUTHORITY = PROJECT / "authority/wr_hbp_final_test_authority.json"
SUMMARY = PROJECT / "analysis/wr_hbp_final_outer_summary.json"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# ------------------------------------------------------------------
# 0. Require the exact prior WR-HBP development project WITH checkpoints.
# ------------------------------------------------------------------
prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-wavelet-residual-hbp-project")
    if p.is_dir()
)
if len(prior_projects) != 1:
    raise RuntimeError(
        "Final confirmation membutuhkan tepat satu prior "
        "'coffee17-wavelet-residual-hbp-project' dari output run WR-HBP lama. "
        "Analysis-package ZIP saja tidak cukup karena best.pt/last.pt wajib ada."
    )

DEV_PROJECT = prior_projects[0]
DEV_ROOT = DEV_PROJECT / "experiments/coffee17-wavelet-residual-hbp-v1"
if not DEV_ROOT.is_dir():
    raise RuntimeError(f"Development experiment root tidak ditemukan: {DEV_ROOT}")

for fold in range(1, 6):
    for arm in ("R0_HBP", "WR_HBP"):
        for name in ("best.pt", "last.pt", "run_contract.json"):
            p = DEV_ROOT / f"fold_{fold}" / "seed42" / arm / name
            if not p.is_file():
                raise RuntimeError(f"Development artifact wajib hilang: {p}")
    if not (DEV_ROOT / f"fold_{fold}" / "seed42" / "pair_result.json").is_file():
        raise RuntimeError(f"pair_result fold {fold} tidak ditemukan.")

print("DEVELOPMENT PROJECT:", DEV_PROJECT)
print("No retraining will be performed.")

# ------------------------------------------------------------------
# 1. Exact final-confirmation code checkout.
# ------------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])
checked_out = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
if checked_out != SCIENTIFIC_CODE_COMMIT:
    raise RuntimeError(f"Checkout salah: {checked_out} != {SCIENTIFIC_CODE_COMMIT}")

run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    REPO / "requirements/preprocessing-study.txt"
])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("DEVELOPMENT_CODE_COMMIT:", DEVELOPMENT_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/experiments/test_wr_hbp_final_confirmation.py",
], cwd=REPO)
print("PRE-TEST CODE CHECKS: PASS")

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "authority").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 2. Reconstruct the exact frozen Coffee17 fold manifest.
#    Still NO outer-test materialization here.
# ------------------------------------------------------------------
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_test import (
    materialize_preprocessing_test,
)

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wr_final_original.zip"
PROV_LOCAL = WORK / "coffee17_wr_final_provenance"
CANONICAL = WORK / "coffee17_wr_final_original_v1"
FOLDS_LOCAL = WORK / "coffee17_wr_final_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Frozen fold reconstruction gagal.")

CLEAN = FOLDS_LOCAL / "clean_manifest.json"
FOLD_MANIFEST = FOLDS_LOCAL / "fold_manifest.json"
CONFIG = REPO / "configs/wr_hbp_final_confirmation/WR_HBP_FINAL_CONFIRMATION_V1.yaml"

# ------------------------------------------------------------------
# 3. PRE-TEST AUTHORITY.
#    This reads ONLY frozen development artifacts + manifests.
#    If it fails, test images are never materialized.
# ------------------------------------------------------------------
if AUTHORITY.exists():
    AUTHORITY.unlink()

run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.build_wr_hbp_final_test_authority",
    "--development-root", DEV_ROOT,
    "--clean-manifest", CLEAN,
    "--fold-manifest", FOLD_MANIFEST,
    "--output", AUTHORITY,
], cwd=REPO, log_path=PROJECT / "logs/authority.log")

authority = json.loads(AUTHORITY.read_text(encoding="utf-8"))
if authority["decision"] != "AUTHORIZE_OOF_TEST_EVALUATION":
    raise RuntimeError("Authority tidak mengizinkan outer test.")
if authority["test_images_accessed"] is not False:
    raise RuntimeError("Authority dibangun setelah test access — STOP.")
if authority["further_primary_tuning_authorized"] is not False:
    raise RuntimeError("Authority masih mengizinkan tuning — STOP.")
if authority["development_scientific_commit"] != DEVELOPMENT_CODE_COMMIT:
    raise RuntimeError("Development scientific commit tidak cocok.")

print("\n=== PRE-TEST AUTHORITY PASS ===")
print("Development gate:", authority["development_gate"])
print("Clean count:", authority["clean_count"])
print("Further tuning authorized:", authority["further_primary_tuning_authorized"])

# ------------------------------------------------------------------
# 4. ONE-SHOT outer test: materialize one fold, infer both frozen arms,
#    then destroy that temporary test runtime.
# ------------------------------------------------------------------
for fold in range(1, 6):
    print(f"\n================ OUTER FOLD {fold}/5 ================", flush=True)
    with tempfile.TemporaryDirectory(
        prefix=f"coffee17_wr_final_test_fold{fold}_", dir=WORK
    ) as tmp:
        TEST_ROOT = Path(tmp)

        contract = materialize_preprocessing_test(
            CANONICAL,
            CLEAN,
            FOLD_MANIFEST,
            AUTHORITY,
            TEST_ROOT,
            fold=fold,
            authorize_test=True,
        )
        if contract["training_executed"] is not False:
            raise RuntimeError("Test contract unexpectedly indicates training.")

        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_wr_hbp_final_outer_fold",
            "--config", CONFIG,
            "--fold", str(fold),
            "--test-root", TEST_ROOT,
            "--development-root", DEV_ROOT,
            "--authority", AUTHORITY,
            "--output-root", OUT,
            "--required-commit", SCIENTIFIC_CODE_COMMIT,
            "--device", "cuda:0",
        ], cwd=REPO, log_path=PROJECT / "logs" / f"outer_fold_{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "fold_result.json").read_text(encoding="utf-8")
    )
    d = result["DELTA_WR_MINUS_R0"]
    print(
        f"OUTER FOLD {fold} | "
        f"R0 Macro={result['R0_HBP']['macro_f1']:.4f} | "
        f"WR Macro={result['WR_HBP']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f}"
    )
    torch.cuda.empty_cache()

# ------------------------------------------------------------------
# 5. Pooled OOF summary + frozen gate + 10,000 paired stratified bootstrap.
# ------------------------------------------------------------------
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_wr_hbp_final_outer_summary",
    "--output-root", OUT,
    "--authority", AUTHORITY,
    "--config", CONFIG,
    "--clean-manifest", CLEAN,
    "--output", SUMMARY,
], cwd=REPO, log_path=PROJECT / "logs/summary.log")

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== FINAL CONFIRMATION ===")
print("Decision:", summary["confirmation_gate"]["decision"])
print("OOF identities:", summary["oof_identity_count"])
print("R0 Macro-F1:", f"{summary['pooled']['R0_HBP']['macro_f1']:.4%}")
print("WR Macro-F1:", f"{summary['pooled']['WR_HBP']['macro_f1']:.4%}")
print("Delta Macro-F1:", f"{summary['delta_wr_minus_r0']['macro_f1']:+.4%}")
print("Delta Hard-F1:", f"{summary['delta_wr_minus_r0']['hard_class_f1']:+.4%}")
print("Delta Worst-F1:", f"{summary['delta_wr_minus_r0']['worst_class_f1']:+.4%}")
print("Positive Macro outer folds:", summary["positive_macro_outer_folds"], "/5")
print(
    "Rescue / damage:",
    summary["paired_prediction_outcomes"]["rescue"],
    "/",
    summary["paired_prediction_outcomes"]["damage"],
)
boot = summary["paired_stratified_bootstrap_macro_delta"]
print(
    "Bootstrap Macro delta 95% CI:",
    f"[{boot['lower_95']:+.4%}, {boot['upper_95']:+.4%}]",
    "| P(delta>0)=", f"{boot['probability_delta_gt_zero']:.3f}",
)
print("Further tuning authorized:", summary["further_tuning_authorized"])

# ------------------------------------------------------------------
# 6. Compact result package. No checkpoints are modified or retrained.
# ------------------------------------------------------------------
PKG = WORK / "wr-hbp-final-confirmation-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copy2(AUTHORITY, PKG / AUTHORITY.name)
shutil.copy2(SUMMARY, PKG / SUMMARY.name)
shutil.copy2(CONFIG, PKG / CONFIG.name)
shutil.copy2(
    REPO / "docs/protocols/WR_HBP_FINAL_OUTER_CONFIRMATION_V1.md",
    PKG / "WR_HBP_FINAL_OUTER_CONFIRMATION_V1.md",
)

for fold in range(1, 6):
    src = OUT / f"fold_{fold}"
    dst = PKG / f"fold_{fold}"
    dst.mkdir(parents=True)
    shutil.copy2(src / "fold_result.json", dst / "fold_result.json")
    for arm in ("R0_HBP", "WR_HBP"):
        arm_src = src / arm
        arm_dst = dst / arm
        arm_dst.mkdir(parents=True)
        for name in ("metrics.json", "predictions.csv"):
            shutil.copy2(arm_src / name, arm_dst / name)

zip_path = Path(shutil.make_archive(
    str(WORK / "wr-hbp-final-confirmation-v1-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("FINAL PROTOCOL CLOSED: no post-test tuning is authorized.")

# Remove temporary reconstruction/code only. Keep PROJECT + result ZIP.
for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
